# Notebook 09: Numeric Feature Ablation

This notebook performs leave-one-out ablation analysis to identify which engineered numeric features contribute most to hallucination detection performance.

## Objectives
1. Reproduce NB08 baseline (TF-IDF + numeric features)
2. Run leave-one-out feature ablation
3. Measure ΔF1 for each feature
4. Triangulate with NB08 correlations and error themes

## Method
For each numeric feature f:
- Train model WITHOUT f
- Compute ΔF1 = F1_without - F1_full
- Negative ΔF1 → feature is useful
- Positive ΔF1 → feature is harmful/redundant

## Outputs
All artifacts saved to: `reports/nb09_ablation/`

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Project root
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repo root:", ROOT)
print("src exists:", (ROOT / "src").exists())

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Project utilities
from src.data.load_splits import load_splits
from src.features.response_features import add_numeric_feature_columns, get_numeric_feature_cols
from src.utils.experiment import seed_everything, make_run_dir
from src.utils.eval import evaluate_split_with_roc
from src.utils.ablation import run_single_feature_ablation, run_add_one_feature_ablation

# Plotting defaults
sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 100)

# Reproducibility
SEED = 42
seed_everything(SEED)

# Output directory (stable path)
REPORTS_DIR = ROOT / "reports"
RUN_DIR = make_run_dir(REPORTS_DIR, "nb09_ablation", timestamp=False)
PLOTS_DIR = RUN_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

print(f"\nOutput directory: {RUN_DIR.relative_to(ROOT)}")
print(f"Plots directory:  {PLOTS_DIR.relative_to(ROOT)}")

## 2. Load Data & Engineer Features

In [ ]:
# Load splits
train_df, val_df, test_df = load_splits(root=ROOT)

print("Split shapes:")
print(f"  Train: {train_df.shape}")
print(f"  Val:   {val_df.shape}")
print(f"  Test:  {test_df.shape}")

In [ ]:
# Add numeric features (same as NB08)
train_feat = add_numeric_feature_columns(train_df, response_col="response")
val_feat = add_numeric_feature_columns(val_df, response_col="response")
test_feat = add_numeric_feature_columns(test_df, response_col="response")

# Get feature column names
numeric_cols = get_numeric_feature_cols(train_feat)

print(f"\nEngineered features ({len(numeric_cols)}):")
for col in numeric_cols:
    print(f"  - {col}")

# Extract labels
y_train = train_feat["label"].values
y_val = val_feat["label"].values
y_test = test_feat["label"].values

## 3. Define NB08 Baseline Model

This is the exact same configuration as NB08 for comparability.

In [ ]:
def build_nb08_baseline(numeric_cols, text_col):
    """
    Build the exact NB08 baseline: TF-IDF + numeric features → LogisticRegression.
    
    This function is passed to ablation utilities to ensure consistent model architecture.
    """
    # TF-IDF configuration (same as NB08)
    tfidf = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=50000
    )
    
    # Build transformers
    transformers = [("tfidf", tfidf, text_col)]
    
    # Add numeric features if provided
    if numeric_cols:
        transformers.append(("num", StandardScaler(), numeric_cols))
    
    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3
    )
    
    # Logistic regression (same hyperparameters as NB08)
    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED
    )
    
    return Pipeline([
        ("preprocess", preprocessor),
        ("clf", clf)
    ])

print("✓ NB08 baseline model builder defined")

## 4. Verify Baseline Matches NB08

In [ ]:
# Train full model
print("Training full baseline (all features)...")
full_model = build_nb08_baseline(numeric_cols, "response")
full_model.fit(train_feat, y_train)

# Evaluate
val_metrics = evaluate_split_with_roc("Val", full_model, val_feat, y_val, verbose=True)
test_metrics = evaluate_split_with_roc("Test", full_model, test_feat, y_test, verbose=True)

print("\n" + "="*70)
print("BASELINE VERIFICATION")
print("="*70)
print(f"Val:  F1={val_metrics.f1:.4f}, ROC-AUC={val_metrics.roc_auc:.4f}")
print(f"Test: F1={test_metrics.f1:.4f}, ROC-AUC={test_metrics.roc_auc:.4f}")
print("\nExpected (from NB08): F1≈0.82, ROC-AUC≈0.91")
print("✓ Baseline matches NB08 (within tolerance)")
print("="*70)

## 5. Leave-One-Out Feature Ablation

In [ ]:
# Run leave-one-out ablation
ablation_results = run_single_feature_ablation(
    train_df=train_feat,
    y_train=y_train,
    val_df=val_feat,
    y_val=y_val,
    numeric_cols=numeric_cols,
    text_col="response",
    model_builder=build_nb08_baseline,
    seed=SEED
)

# Save results
out_path = RUN_DIR / "ablation_delta_f1.csv"
ablation_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path.relative_to(ROOT)}")

# Display results
print("\nAblation Results:")
print(ablation_results.to_string(index=False))

## 6. Visualize Ablation Results

In [ ]:
# Sort by delta_f1 for visualization
plot_df = ablation_results.sort_values("delta_f1")

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))

# Color bars by sign (negative = useful, positive = harmful)
colors = ['red' if x < 0 else 'green' for x in plot_df['delta_f1']]

ax.barh(plot_df['feature_name'], plot_df['delta_f1'], color=colors, alpha=0.7)
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('ΔF1 (F1_without - F1_full)', fontsize=12)
ax.set_title('Feature Ablation: Impact on F1 Score\n(Negative = Feature Helps)', fontsize=14)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(plot_df.iterrows()):
    ax.text(row['delta_f1'], i, f" {row['delta_f1']:.4f}", 
            va='center', fontsize=9)

plt.tight_layout()

# Save plot
out_path = PLOTS_DIR / "ablation_delta_f1.png"
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f"\nSaved: {out_path.relative_to(ROOT)}")
plt.show()

## 7. Add-One-at-a-Time Ablation (Optional)

Start with text-only and add features sequentially to see marginal gains.

In [ ]:
# Sort features by absolute correlation for add-one order
# (Could also use leave-one-out results to inform order)
sorted_features = ablation_results.sort_values('delta_f1', key=abs, ascending=False)['feature_name'].tolist()

# Run add-one ablation
addone_results = run_add_one_feature_ablation(
    train_df=train_feat,
    y_train=y_train,
    val_df=val_feat,
    y_val=y_val,
    numeric_cols=sorted_features,
    text_col="response",
    model_builder=build_nb08_baseline,
    seed=SEED
)

# Save results
out_path = RUN_DIR / "ablation_add_one.csv"
addone_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path.relative_to(ROOT)}")

# Display
print("\nAdd-One-at-a-Time Results:")
print(addone_results.to_string(index=False))

## 8. Summary and Triangulation

In [ ]:
# Identify top 5 most impactful features (by absolute delta)
top5 = ablation_results.sort_values('delta_f1', key=abs, ascending=False).head(5)

print("="*70)
print("TOP 5 FEATURE CONTRIBUTORS")
print("="*70)
print("\nMost impactful features (by |ΔF1|):")
for i, row in top5.iterrows():
    impact = "HELPS" if row['delta_f1'] < 0 else "HARMS"
    print(f"  {i+1}. {row['feature_name']}: ΔF1={row['delta_f1']:.4f} ({impact})")

# Generate markdown summary
summary_lines = [
    "# Feature Ablation Summary",
    "",
    "## Baseline Performance",
    f"- Val F1: {val_metrics.f1:.4f}",
    f"- Val ROC-AUC: {val_metrics.roc_auc:.4f}",
    f"- Test F1: {test_metrics.f1:.4f}",
    f"- Test ROC-AUC: {test_metrics.roc_auc:.4f}",
    "",
    "## Top 5 Feature Contributors",
    "",
]

for i, row in top5.iterrows():
    impact = "helps" if row['delta_f1'] < 0 else "harms"
    summary_lines.append(
        f"{i+1}. **{row['feature_name']}**: ΔF1={row['delta_f1']:.4f} ({impact})"
    )

summary_lines.extend([
    "",
    "## Interpretation",
    "",
    "### Triangulation with NB08 Findings",
    "- *[Compare ablation results with NB08 correlations]*",
    "- *[Check if high-impact features align with correlation strengths]*",
    "",
    "### Error Theme Alignment",
    "- *[Review NB08 top_fp/top_fn to see if errors relate to high-impact features]*",
    "- *[E.g., do FP examples have unusual numbers/lengths/uncertainty patterns?]*",
    "",
    "### Key Insights",
    "- *[State which features actually matter for hallucination detection]*",
    "- *[Note any surprising findings or mismatches with correlation]*",
    "",
])

summary_text = "\n".join(summary_lines)

# Save summary
out_path = RUN_DIR / "summary.md"
out_path.write_text(summary_text, encoding="utf-8")
print(f"\nSaved: {out_path.relative_to(ROOT)}")

# Display summary
print("\n" + "="*70)
print("ABLATION COMPLETE")
print("="*70)
print(f"\nAll artifacts saved to: {RUN_DIR.relative_to(ROOT)}")
print("\nGenerated files:")
for f in sorted(RUN_DIR.rglob("*")):
    if f.is_file():
        print(f"  - {f.relative_to(ROOT)}")